# 학생 성취도 예측 — 전처리 파이프라인

**순서**: 드롭 → 결측치 처리 → Ordinal Encoding → One-Hot Encoding → 스케일링 → 저장  
**학번**: 20242530 정명진

---
## 0. 환경 설정

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         'archive', 'hybrid_student_performance_1200.csv')
OUT_PATH  = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         'archive', 'preprocessed_student_performance.csv')

df_raw = pd.read_csv(DATA_PATH)
print(f'원본: {df_raw.shape[0]}행 × {df_raw.shape[1]}열')
df_raw.head(3)

---
## 1. 불필요 컬럼 드롭

In [ ]:
DROP_COLS = ['student_id', 'timestamp']

df = df_raw.drop(columns=DROP_COLS)
print(f'드롭 후: {df.shape[0]}행 × {df.shape[1]}열  (제거: {DROP_COLS})')

---
## 2. 결측치 처리

In [ ]:
# 결측치 현황
missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)
print('결측치 있는 컬럼:')
print(missing_before.to_frame('결측 수').assign(결측률=lambda x: (x['결측 수']/len(df)*100).round(2)))

In [ ]:
# 수치형 → 중앙값, 범주형 → 최빈값 대체
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  [수치형] {col}: 중앙값({median_val}) 대체')

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'  [범주형] {col}: 최빈값("{mode_val}") 대체')

print(f'\n결측치 처리 후 잔여 결측: {df.isnull().sum().sum()}개')

---
## 3. Ordinal Encoding (순서형 범주 → 정수)

In [ ]:
ORDINAL_MAPS = {
    # 만족도
    'academic_satisfaction': {
        'Very unsatisfied': 1, 'Unsatisfied': 2, 'Neutral': 3, 'Satisfied': 4, 'Very satisfied': 5
    },
    # 공부 시간
    'study_hours_daily': {
        'Less than 1 hour': 0, '1\u20132 hours': 1, 'More than 2 hours': 2
    },
    # 복습 빈도
    'revision_frequency': {
        'Never': 0, 'Rarely': 1, 'Few times a week': 2, 'Daily': 3
    },
    # 집중 시간
    'focus_duration': {
        '30\u201360 minutes': 0, '1\u20132 hours': 1, 'More than 2 hours': 2
    },
    # 비학습 스크린 타임
    'screen_time_non_study': {
        '2\u20134 hours': 0, '4\u20136 hours': 1, 'More than 6 hours': 2
    },
    # 공부 일관성
    'study_consistency': {
        'Rarely': 0, 'Sometimes': 1, 'Mostly consistent': 2
    },
    # 과제 제때 제출
    'tasks_on_time': {
        'Rarely': 0, 'Sometimes': 1, 'Often': 2, 'Always': 3
    },
    # 수업 중 졸림
    'sleepy_during_study': {
        'Never': 0, 'Sometimes': 1, 'Often': 2, 'Always': 3
    },
    # 수면 시간
    'sleep_hours': {
        '4\u20135 hours': 0, '6\u20137 hours': 1, 'More than 8 hours': 2
    },
    # 과제 제출
    'assignments_on_time': {
        'Rarely': 0, 'Sometimes': 1, 'Often': 2, 'Always': 3
    },
    # 출석률
    'attendance_percentage': {
        'Less than 50%': 0, '50% \u2013 65%': 1, '66% \u2013 75%': 2,
        '76% \u2013 85%': 3, 'Above 85%': 4
    },
    # 외부 자료 활용
    'external_resources': {
        'Never (Unaware or Not interested)': 0,
        'Rarely (Passive)': 1,
        'Occasionally (When needed)': 2
    },
    # 외부 압박
    'external_pressure': {
        'No Impact (Fully supportive environment)': 0,
        'Low Impact (Rarely affects study)': 1,
        'Moderate Impact (Occasional disruption)': 2,
        'High Impact (Frequent disruption)': 3
    },
    # 진로 목표 명확도
    'career_goal_clarity': {
        'Not clear': 0, 'Somewhat clear': 1, 'Very clear': 2
    },
    # 프로그래밍 기반
    'programming_foundation': {
        'Limited knowledge, theoretical only': 0,
        'Basic knowledge, learning while practicing': 1,
        'Strong foundation in core concepts': 2
    },
    # 행사 참여
    'events_participation': {
        'Never participate in such events': 0,
        'Rarely participate, mostly observe': 1,
        'Occasionally participate in events': 2
    },
    # 타겟 ①: 위험도
    'performance_risk_level': {
        'Low Risk': 0, 'Moderate Risk': 1, 'High Risk': 2
    },
    # 타겟 ②: CGPA 구간
    'cgpa_category': {
        '5.0 \u2013 6.9': 0, '7.0 \u2013 8.4': 1,
        '8.5 \u2013 9.4': 2, '9.5 \u2013 10.0': 3
    },
}

for col, mapping in ORDINAL_MAPS.items():
    if col in df.columns:
        before_unique = df[col].unique()
        df[col] = df[col].map(mapping)
        unmapped = df[col].isnull().sum()
        status = f'OK (범위 {df[col].min()}~{df[col].max()})' if unmapped == 0 else f'주의: {unmapped}개 미매핑'
        print(f'  {col}: {status}')

print(f'\nOrdinal Encoding 완료: {len(ORDINAL_MAPS)}개 컬럼')

---
## 4. One-Hot Encoding (명목형 범주)

In [ ]:
OHE_COLS = [
    'year_class', 'program_stream', 'gender',
    'main_distractor', 'skills_developing',
    'career_interest', 'online_courses',
    'projects_internships', 'preparation_status',
    'strongest_asset', 'internal_barrier'
]

# 실제 존재하는 컬럼만 OHE 적용
ohe_cols_exist = [c for c in OHE_COLS if c in df.columns]
cols_before = df.shape[1]

df = pd.get_dummies(df, columns=ohe_cols_exist, drop_first=False, dtype=int)

cols_after = df.shape[1]
print(f'OHE 적용: {ohe_cols_exist}')
print(f'컬럼 수 변화: {cols_before} → {cols_after} (+{cols_after - cols_before}개)')

---
## 5. 수치형 스케일링 (StandardScaler)

In [ ]:
# 타겟 컬럼은 스케일링 제외
TARGET_COLS = ['cgpa_category', 'performance_risk_level']
SCALE_COLS  = ['age', 'daily_productivity', 'energy_level', 'stress_level', 'routine_rating']
scale_exist = [c for c in SCALE_COLS if c in df.columns]

scaler = StandardScaler()
df[scale_exist] = scaler.fit_transform(df[scale_exist])

print(f'스케일링 완료: {scale_exist}')
df[scale_exist].describe().T[['mean', 'std', 'min', 'max']].round(3)

---
## 6. 전처리 결과 확인

In [ ]:
print(f'최종 데이터 형태: {df.shape[0]}행 × {df.shape[1]}열')
print(f'잔여 결측치: {df.isnull().sum().sum()}개')
print('\n컬럼 목록:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:3d}. {col}')

In [ ]:
# 타겟 변수 분포 확인 (클래스 불균형)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, title, labels in zip(
    axes,
    ['cgpa_category', 'performance_risk_level'],
    ['CGPA 구간 (0~3)', '위험도 (0=Low, 1=Moderate, 2=High)'],
    [['5.0-6.9', '7.0-8.4', '8.5-9.4', '9.5-10.0'], ['Low', 'Moderate', 'High']]
):
    cnt = df[col].value_counts().sort_index()
    colors = sns.color_palette('muted', len(cnt))
    bars = ax.bar([str(x) for x in cnt.index], cnt.values, color=colors, edgecolor='white')
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 3,
                int(b.get_height()), ha='center', fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('학생 수')
    ax.set_xlabel('레이블')

plt.suptitle('타겟 클래스 분포 확인 (전처리 후)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 전처리 전·후 요약
summary = pd.DataFrame({
    '항목': ['행 수', '컬럼 수', '결측치 합계', '범주형 컬럼', '수치형 컬럼'],
    '전처리 전': [
        df_raw.shape[0], df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        df_raw.select_dtypes('object').shape[1],
        df_raw.select_dtypes('number').shape[1]
    ],
    '전처리 후': [
        df.shape[0], df.shape[1],
        df.isnull().sum().sum(),
        df.select_dtypes('object').shape[1],
        df.select_dtypes('number').shape[1]
    ]
})
summary.set_index('항목')

---
## 7. 전처리 데이터 저장

In [ ]:
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUT_PATH}')
print(f'파일 크기: {os.path.getsize(OUT_PATH) / 1024:.1f} KB')
df.head(3)